# v10 soft: fold 0-2, nhãn 2026-10-29b-doc

Dữ liệu: dataset Kaggle `ducanhpham2k5/p3-log-data` . Kết quả đẩy lên dataset riêng tư `ducanhpham2k5/p3-colab-v10-doc-a` sau mỗi fold.

Cần: Runtime GPU (L4), secret `KAGGLE_API_TOKEN` (biểu tượng chìa khoá). Chạy lần lượt các ô; ô 3 chạy nền, ô 4 xem tiến độ. Giữ tab mở đến khi log có dòng XONG.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
import os, json, subprocess
import pandas as pd
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
subprocess.run("pip install -q -U kaggle", shell=True)
D, OUT, SLUG = "/content/p3data", "/content/out", "p3-colab-v10-doc-a"
os.makedirs(OUT, exist_ok=True)
r = subprocess.run(f"kaggle datasets download ducanhpham2k5/p3-log-data -p {D} --unzip", shell=True, capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-300:])
runs = pd.read_parquet(f"{D}/runs.parquet")
print(len(runs), "run | crash:", (runs.leaf == "crash").sum(), "| quality:", (runs.leaf == "quality").sum())
assert (runs.leaf == "doc").sum() > 280 and (runs.leaf == "crash").sum() > 3010, "dataset vẫn là nhãn cũ: chờ bản p3-log-data mới"
json.dump({"title": SLUG, "id": f"ducanhpham2k5/{SLUG}", "licenses": [{"name": "other"}]},
          open(f"{OUT}/dataset-metadata.json", "w"))

In [ ]:
script = """
export PYTHONPATH=D/project3_code.pyz PYTHONIOENCODING=utf-8
up() { kaggle datasets version -p OUT -m "$1" -r zip -q || kaggle datasets create -p OUT -r zip -q; }
for f in 0 1 2; do
  echo "=== fold $f $(date)" >> OUT/train.log
  python -m project3.model.train --data {D} --fold $f --variants soft --feats all --chunks error --max-tokens 512 --window 1600 --out OUT/results_fold$f.json >> OUT/train.log 2>&1 || echo "fold $f LOI" >> OUT/train.log
  up "fold $f" >> OUT/upload.log 2>&1
done
echo "XONG $(date)" >> OUT/train.log
""".replace("OUT", OUT).replace("D/", D + "/").replace("{D}", D)
open("/content/run.sh", "w").write(script)
proc = subprocess.Popen(["bash", "/content/run.sh"])
print("pid", proc.pid)

In [ ]:
import re
log = open(f"{OUT}/train.log", encoding="utf-8", errors="replace").read().splitlines()
keep = [l for l in log if re.search(r"===|test full|test masked|epoch|step \d+000 |LOI|XONG|Traceback|Error", l)]
print("đang chạy" if proc.poll() is None else f"đã dừng, mã {proc.returncode}")
print("\n".join(keep[-15:]))
print(sorted(os.listdir(OUT)))